۔'، /!unzip /content/skindiseasedataset.zip -d /content/

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Higher resolution images for better feature extraction
IMG_SIZE = 160  # Increased from 128

# Enhanced data augmentation settings
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,  # Skin lesions can appear in any orientation
    zoom_range=0.3,
    shear_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

# Validation generator with only rescaling
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training data with augmentation
train_generator = train_datagen.flow_from_directory(
    r'/content/skindiseasedataset',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,  # Smaller batch size
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation data without augmentation
validation_generator = validation_datagen.flow_from_directory(
    r'/content/skindiseasedataset',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Print dataset information
print(f"Number of classes: {len(train_generator.class_indices)}")
print(f"Class names: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

# Build an optimized CNN architecture
cnn = Sequential([
    # First block
    Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 3),
           kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Second block
    Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Third block - deeper network
    Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(256, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Fourth block - even deeper
    Conv2D(512, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(512, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Dropout to reduce overfitting
    Dropout(0.4),

    # Flatten and dense layers
    Flatten(),
    Dense(1024, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(len(train_generator.class_indices), activation='softmax')
])

# Model summary
cnn.summary()

# Training callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_cnn_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Compile model with optimized learning rate
cnn.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Number of epochs - allow for more training time with early stopping
epochs = 100

# Train the model
print("Training CNN model...")
history = cnn.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=epochs,
    callbacks=[early_stopping, lr_scheduler, checkpoint]
)

# Load best model from checkpoint
cnn = tf.keras.models.load_model('best_cnn_model.h5')
print("Loaded best model from checkpoint")

# Save final model
cnn.save('skin_disease_cnn_final.h5')
print("Final model saved successfully as skin_disease_cnn_final.h5")

# Evaluate model
print("Evaluating model on training data...")
train_loss, train_acc = cnn.evaluate(train_generator)
print("Evaluating model on validation data...")
val_loss, val_acc = cnn.evaluate(validation_generator)
print(f'Training Accuracy: {train_acc:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}')

# Accuracy/Loss Graphs
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

# Reset generator
validation_generator.reset()

# Confusion Matrix
print("Generating confusion matrix...")
true_labels = []
predictions = []
batch_count = 0
max_batches = validation_generator.samples // validation_generator.batch_size + 1

for i, (images, labels) in enumerate(validation_generator):
    batch_count += 1
    preds = cnn.predict(images, verbose=0)
    true_labels.extend(np.argmax(labels, axis=1))
    predictions.extend(np.argmax(preds, axis=1))
    if batch_count >= max_batches:
        break

# Convert to arrays for metrics calculations
true_labels = np.array(true_labels)
predictions = np.array(predictions)

# Get class names
class_names = list(validation_generator.class_indices.keys())

# Create confusion matrix
cm = confusion_matrix(true_labels, predictions, normalize='true')

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(true_labels, predictions, target_names=class_names))

# ROC Curve
print("Generating ROC curves...")
validation_generator.reset()

# Collect true labels and predictions for ROC curve
y_true = []
y_pred = []
batch_count = 0

for i, (images, labels) in enumerate(validation_generator):
    batch_count += 1
    pred_batch = cnn.predict(images, verbose=0)
    y_true.append(labels)
    y_pred.append(pred_batch)
    if batch_count >= max_batches:
        break

# Concatenate batches
y_true = np.vstack(y_true)
y_pred = np.vstack(y_pred)

# Plot ROC curve for each class
plt.figure(figsize=(14, 12))

for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_true[:, i], y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

# Add diagonal line
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for All Classes')
plt.legend(loc='lower right', fontsize='small')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('roc_curves.png')
plt.show()

In [ ]:
!unzip /content/skindiseasedataset.zip -d /content/

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing import image
from sklearn.metrics import confusion_matrix, classification_report

# Load the trained model
model = tf.keras.models.load_model('/content/best_cnn_model.h5')
print("Model loaded successfully!")

# Path to the test dataset
test_images_path = "/content/testing"  # Change this to your actual test folder path

# Get class labels (folder names)
class_labels = sorted(os.listdir(test_images_path))  # Sort to match training order

# Initialize lists for true and predicted labels
true_labels = []
predicted_labels = []

# Loop through each class folder
for class_name in class_labels:
    class_folder = os.path.join(test_images_path, class_name)
    if not os.path.isdir(class_folder):
        continue  # Skip non-folder files

    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)

        # Load and preprocess image
        # Use the same IMG_SIZE as during training (160)
        img = image.load_img(img_path, target_size=(160, 160))  # Resize to match training size
        img_array = image.img_to_array(img) / 255.0  # Normalize
        img_array = np.expand_dims(img_array, axis=0)  # Expand dimensions

        # Predict class
        predictions = model.predict(img_array)
        predicted_class_idx = np.argmax(predictions)

        # Store actual and predicted labels
        true_labels.append(class_labels.index(class_name))  # Convert to index
        predicted_labels.append(predicted_class_idx)

        # Display Image and Prediction (Optional)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Actual: {class_name} | Predicted: {class_labels[predicted_class_idx]}")
        plt.show()

# Generate Confusion Matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Print Classification Report
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=class_labels))